# 3.21 — Naive Bayes

Naive Bayes is a probabilistic classifier that scores each class by multiplying a prior belief by feature likelihoods, then normalizing those scores into posterior probabilities. In this lesson, we build the whole machine from NumPy arrays: class priors, categorical likelihood tables, Laplace smoothing, log-space products, Gaussian likelihoods, and validation-aware model selection.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Naive Bayes one idea at a time. Run each cell in order and read the printed intermediate values — every probability, product, normalization, and stability trick is shown directly. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, counting, logs, exponentials, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any randomized demonstration.

### 1. Turning labels into class priors

A classifier begins with a prior belief about each class before seeing any features. In Naive Bayes, that prior is the class frequency in the training set: if 3 of 8 messages are spam, then the model starts from `p(spam)=3/8` before reading any words.

In [ ]:
y_w = np.array([0, 0, 0, 0, 0, 1, 1, 1])  # 0 = ham, 1 = spam.
classes_w, counts_w = np.unique(y_w, return_counts=True)
priors_w = counts_w / counts_w.sum()
print("classes:", classes_w)
print("class counts:", counts_w)
print("priors:", np.round(priors_w, 3))
assert np.allclose(priors_w, [0.625, 0.375])

▶ What you'll see: ham is more common than spam, so the prior starts at 0.625 versus 0.375.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["ham", "spam"], priors_w, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.ylabel("prior probability")
plt.title("1: class priors from label counts")
plt.show()

▶ What you'll see: the prior bar for ham is taller; a spam-like feature must compensate for that base-rate disadvantage.

*Why it's done this way:* Bayes' rule needs `p(y)` before it can update to `p(y|x)`. Estimating `p(y)` by relative frequency makes the model respect the empirical class balance instead of pretending every class is equally likely.

### 2. Feature likelihoods from counts

Now we ask how often each feature appears inside each class. For binary word features, `p(x_j=1|y=k)` is the fraction of training examples in class `k` where word `j` is present.

In [ ]:
X_w = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]])
words_w = np.array(["meeting", "win", "lunch"])
ham_rows_w = X_w[y_w == 0]
spam_rows_w = X_w[y_w == 1]
print("X shape:", X_w.shape)
print("word columns:", words_w)

▶ What you'll see: 8 examples by 3 binary word features.

In [ ]:
raw_lik_w = np.vstack([ham_rows_w.mean(axis=0), spam_rows_w.mean(axis=0)])
print("p(word present | class):\n", np.round(raw_lik_w, 3))
assert np.allclose(np.round(raw_lik_w, 3), [[0.6, 0.0, 0.8], [0.333, 1.0, 0.333]])

▶ What you'll see: `win` is never seen in ham and always seen in spam in this tiny sample.

In [ ]:
xpos_w = np.arange(len(words_w))
plt.figure(figsize=(5, 3))
plt.bar(xpos_w - 0.18, raw_lik_w[0], width=0.36, label="ham", color="steelblue")
plt.bar(xpos_w + 0.18, raw_lik_w[1], width=0.36, label="spam", color="crimson")
plt.xticks(xpos_w, words_w); plt.ylim(0, 1.05)
plt.ylabel("p(word present | class)"); plt.title("2: feature likelihoods")
plt.legend(); plt.show()

▶ What you'll see: `win` is the most spam-indicative feature, while `lunch` is more ham-like.

*Why it's done this way:* A likelihood table is the model's memory. It stores how each feature behaves under each class, because Bayes' rule compares how plausible the observed feature vector would be if each class were true.

### 3. The naive conditional-independence product

Naive Bayes becomes simple by assuming features are conditionally independent once the class is known. That means the joint likelihood decomposes into a product over feature likelihoods.

In [ ]:
x_new_w = np.array([1, 1, 0])  # meeting present, win present, lunch absent.
ham_terms_w = np.where(x_new_w == 1, raw_lik_w[0], 1 - raw_lik_w[0])
spam_terms_w = np.where(x_new_w == 1, raw_lik_w[1], 1 - raw_lik_w[1])
print("new message:", dict(zip(words_w, x_new_w)))
print("ham terms:", np.round(ham_terms_w, 3))
print("spam terms:", np.round(spam_terms_w, 3))

▶ What you'll see: each present feature uses `p(feature present|class)` and absent lunch uses its complement.

In [ ]:
print("ham product:", np.prod(ham_terms_w))
print("spam product:", np.prod(spam_terms_w))
assert np.prod(ham_terms_w) == 0.0

▶ What you'll see: the ham product is exactly 0 because `win` never appeared in ham.

*Why it's done this way:* The product is the conditional-independence assumption in arithmetic form. We multiply because each feature is treated as a separate piece of evidence given the class; the assumption is usually false in detail, but it often gives a useful low-variance classifier.

### 4. Laplace smoothing prevents zero-probability collapse

A single unseen feature should not make a class impossible forever. Laplace smoothing adds a small pseudocount to every feature outcome before dividing, so every class keeps nonzero probability for every feature.

In [ ]:
alpha_w = 1.0
present_counts_w = np.vstack([ham_rows_w.sum(axis=0), spam_rows_w.sum(axis=0)])
class_sizes_w = np.array([len(ham_rows_w), len(spam_rows_w)])
smoothed_lik_w = (present_counts_w + alpha_w) / (class_sizes_w[:, None] + 2 * alpha_w)
print("present counts:\n", present_counts_w)
print("smoothed likelihoods:\n", np.round(smoothed_lik_w, 3))
assert np.allclose(np.round(smoothed_lik_w, 3), [[0.571, 0.143, 0.714], [0.4, 0.8, 0.4]])

▶ What you'll see: the ham probability for `win` rises from 0 to 0.143.

In [ ]:
ham_terms_s_w = np.where(x_new_w == 1, smoothed_lik_w[0], 1 - smoothed_lik_w[0])
spam_terms_s_w = np.where(x_new_w == 1, smoothed_lik_w[1], 1 - smoothed_lik_w[1])
class_scores_w = priors_w * np.array([np.prod(ham_terms_s_w), np.prod(spam_terms_s_w)])
posterior_w = class_scores_w / class_scores_w.sum()
print("unnormalized scores:", np.round(class_scores_w, 5))
print("posterior p(y|x):", np.round(posterior_w, 3))
assert np.allclose(np.round(posterior_w, 3), [0.168, 0.832])

▶ What you'll see: spam wins, but ham is no longer mathematically erased.

*Why it's done this way:* Smoothing is regularization for probability tables. It trades a little training fit for robustness, especially when a small dataset has accidental zeros that would otherwise dominate every future prediction.

### 5. Normalization turns scores into posteriors

Bayes' rule first gives proportional scores: `p(y=k)p(x|y=k)`. Those scores do not have to sum to 1, so we divide by their sum across classes.

In [ ]:
score_names_w = np.array(["ham", "spam"])
print("raw scores:", dict(zip(score_names_w, np.round(class_scores_w, 5))))
print("score sum:", round(float(class_scores_w.sum()), 5))

▶ What you'll see: the raw scores are tiny likelihood-weighted priors, not probabilities over labels yet.

In [ ]:
posterior_check_w = class_scores_w / np.sum(class_scores_w)
print("normalized posterior:", dict(zip(score_names_w, np.round(posterior_check_w, 3))))
print("posterior sum:", round(float(posterior_check_w.sum()), 3))
assert round(float(posterior_check_w.sum()), 3) == 1.0

▶ What you'll see: the two posterior probabilities sum to 1 exactly up to rounding.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(score_names_w, posterior_check_w, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.ylabel("posterior probability")
plt.title("5: normalized Naive Bayes posterior")
plt.show()

▶ What you'll see: the spam posterior is higher because the smoothed evidence for `win` overwhelms the ham prior.

*Why it's done this way:* The denominator is the total probability of seeing this feature vector under all class explanations. Dividing by it puts every class on the same probability scale.

### 6. Log-space products keep long feature vectors stable

Multiplying many probabilities can underflow toward zero. Taking logs converts products into sums, so we compare log-scores directly and exponentiate only after subtracting the maximum log-score.

In [ ]:
long_probs_w = np.full(300, 0.02)
plain_product_w = float(np.prod(long_probs_w))
log_sum_w = float(np.sum(np.log(long_probs_w)))
print("plain product:", plain_product_w)
print("log product:", round(log_sum_w, 3))
assert plain_product_w == 0.0

▶ What you'll see: the direct product underflows to 0, but the log-sum remains finite.

In [ ]:
log_lik_present_w = np.log(smoothed_lik_w)
log_lik_absent_w = np.log(1 - smoothed_lik_w)
log_scores_w = np.log(priors_w) + np.sum(np.where(x_new_w == 1, log_lik_present_w, log_lik_absent_w), axis=1)
stable_log_w = np.exp(log_scores_w - np.max(log_scores_w))
posterior_log_w = stable_log_w / stable_log_w.sum()
print("log scores:", np.round(log_scores_w, 3))
print("stable posterior:", np.round(posterior_log_w, 3))
assert np.allclose(posterior_log_w, posterior_w)

▶ What you'll see: log-space prediction matches the earlier posterior without risking numerical underflow.

*Why it's done this way:* Logs preserve the ordering of positive scores while turning fragile multiplication into stable addition. The max-subtraction step rescales exponentials into a safe numerical range without changing the normalized posterior.

### 7. Gaussian Naive Bayes for continuous features

When features are continuous, we replace categorical counts with a density. Gaussian Naive Bayes estimates a mean and variance for each feature inside each class, then evaluates the normal density for each observed feature value.

In [ ]:
Xg_w = np.array([[1.0, 2.0], [1.2, 1.8], [0.8, 2.2], [3.0, 0.8], [3.2, 1.0], [2.8, 0.7]])
yg_w = np.array([0, 0, 0, 1, 1, 1])
means_w = np.vstack([Xg_w[yg_w == k].mean(axis=0) for k in [0, 1]])
vars_w = np.vstack([Xg_w[yg_w == k].var(axis=0) + 1e-6 for k in [0, 1]])
print("means:\n", np.round(means_w, 3))
print("variances:\n", np.round(vars_w, 3))
assert np.allclose(np.round(means_w, 3), [[1.0, 2.0], [3.0, 0.833]])

▶ What you'll see: class 0 clusters near low feature 0 and high feature 1; class 1 clusters in the opposite direction.

In [ ]:
xg_new_w = np.array([2.9, 0.9])
log_gauss_w = -0.5 * (np.log(2 * np.pi * vars_w) + ((xg_new_w - means_w) ** 2) / vars_w)
log_scores_g_w = np.log([0.5, 0.5]) + log_gauss_w.sum(axis=1)
prob_g_w = np.exp(log_scores_g_w - np.max(log_scores_g_w)); prob_g_w = prob_g_w / prob_g_w.sum()
print("Gaussian log scores:", np.round(log_scores_g_w, 3))
print("posterior:", np.round(prob_g_w, 3))
assert prob_g_w[1] > 0.999

▶ What you'll see: the new point is overwhelmingly classified as class 1 because it lies near that class's Gaussian center.

In [ ]:
plt.figure(figsize=(4.5, 3.4))
plt.scatter(Xg_w[yg_w == 0, 0], Xg_w[yg_w == 0, 1], label="class 0", color="steelblue")
plt.scatter(Xg_w[yg_w == 1, 0], Xg_w[yg_w == 1, 1], label="class 1", color="crimson")
plt.scatter([xg_new_w[0]], [xg_new_w[1]], marker="*", s=180, color="black", label="new x")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.title("7: Gaussian NB density intuition")
plt.legend(); plt.show()

▶ What you'll see: the star lands inside the class-1 cluster, matching the posterior calculation.

*Why it's done this way:* Continuous features cannot be counted as exact repeated values, so the Gaussian assumption supplies a smooth likelihood. Naive Bayes still assumes conditional independence across feature dimensions.

### 8. Empirical score, cost, gap, and stabilization

The lesson's prose also emphasizes that model selection is not raw fit alone. We compute an empirical average loss, add a cost term, compare a flexible alternative, and inspect a stabilized score.

In [ ]:
losses_w = np.array([0.213, 0.109, 0.454])
empirical_w = float(losses_w.mean())
cost_w = 0.080
score_w = empirical_w + cost_w
print("empirical risk:", round(empirical_w, 3))
print("score with cost:", round(score_w, 3))
assert round(empirical_w, 3) == 0.259
assert round(score_w, 3) == 0.339

▶ What you'll see: the raw average is 0.259, but the decision score is 0.339 after cost.

In [ ]:
alt_w = 0.379
gap_w = alt_w - score_w
relative_gap_w = gap_w / alt_w
stable_score_w = 0.80 * score_w
choices_w = np.array([score_w, alt_w, stable_score_w])
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stable score:", round(stable_score_w, 3), "best:", round(float(np.min(choices_w)), 3))
assert round(gap_w, 3) == 0.040
assert round(relative_gap_w, 3) == 0.106
assert round(stable_score_w, 3) == 0.271

▶ What you'll see: the stabilized score is the smallest of the three options.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["baseline+cost", "flexible alt", "stabilized"], choices_w, color=["steelblue", "orange", "seagreen"])
plt.ylabel("decision score (lower is better)"); plt.xticks(rotation=15)
plt.title("8: selection uses the full score")
plt.show()

▶ What you'll see: the stabilized option wins after accounting for the cost and comparison scale.

*Why it's done this way:* Validation and regularization live on the same decision scale as the model score. A lower raw training loss can be a mirage if its flexibility cost or validation uncertainty is larger than the apparent gain.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, counting, logs, exponentials, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for the heatmaps, bars, lines, and scatter plots used to inspect Naive Bayes.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def normalize(scores): # turn positive class scores into probabilities that sum to one.
    scores = np.asarray(scores, dtype=float) # convert input to a float array for safe division.
    total = np.sum(scores) # compute the normalizing denominator across classes.
    return scores / total if total > 0 else np.ones_like(scores) / len(scores) # fall back to uniform only if all evidence vanished.

def binary_nb_tables(X, y, alpha=1.0): # estimate priors and Bernoulli likelihoods with Laplace smoothing.
    X = np.asarray(X, dtype=float) # ensure binary feature arithmetic uses floats.
    y = np.asarray(y, dtype=int) # ensure labels can index rows.
    classes = np.unique(y) # collect the class labels present in the data.
    priors = np.array([np.mean(y == k) for k in classes]) # estimate p(y=k) by relative frequency.
    counts = np.vstack([X[y == k].sum(axis=0) for k in classes]) # count feature-presence events within each class.
    sizes = np.array([np.sum(y == k) for k in classes]) # count examples per class.
    probs = (counts + alpha) / (sizes[:, None] + 2 * alpha) # smooth binary p(x_j=1|y=k).
    return classes, priors, probs # return everything needed for prediction.

def binary_nb_predict(x, priors, probs): # compute a Bernoulli Naive Bayes posterior for one binary vector.
    x = np.asarray(x, dtype=float) # convert the feature vector to floats.
    terms = np.where(x == 1, probs, 1 - probs) # choose present or absent likelihood for every class-feature pair.
    scores = priors * np.prod(terms, axis=1) # multiply prior by all conditional likelihoods.
    return normalize(scores) # return normalized posterior probabilities.

def binary_nb_predict_log(x, priors, probs): # compute the same posterior in stable log space.
    x = np.asarray(x, dtype=float) # convert the feature vector to floats.
    log_terms = np.where(x == 1, np.log(probs), np.log(1 - probs)) # choose log likelihoods for present/absent features.
    log_scores = np.log(priors) + np.sum(log_terms, axis=1) # add logs instead of multiplying probabilities.
    shifted = np.exp(log_scores - np.max(log_scores)) # exponentiate after max subtraction for stability.
    return shifted / np.sum(shifted) # normalize the shifted scores into a posterior.

## 🟢 Basics (warm-up)

### Basic 1 — Count class priors

**Goal.** Estimate class priors from label frequencies, because Naive Bayes starts with the base rate before reading any feature. We build it in 2 steps.

In [ ]:
y_b1 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # store toy labels where 0 means ham and 1 means spam.
classes_b1, counts_b1 = np.unique(y_b1, return_counts=True) # count examples per class.
print("classes:", classes_b1) # inspect label order.
print("counts:", counts_b1) # inspect raw class counts.

▶ What you'll see: class 0 has five examples and class 1 has three.

In [ ]:
priors_b1 = counts_b1 / counts_b1.sum() # divide each count by the dataset size to estimate p(y).
print("priors:", np.round(priors_b1, 3)) # inspect the class base rates.
assert np.allclose(priors_b1, [0.625, 0.375]) # verify the canonical prior values.
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], priors_b1, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.title("Basic 1: class priors"); plt.ylabel("p(y)"); plt.show()

▶ What you'll see: ham starts with the larger prior probability.

👀 Takeaway: priors encode class imbalance before feature evidence is considered.

### Basic 2 — Count one word likelihood

**Goal.** Estimate `p(win=1|y)`, because a single word likelihood is the smallest evidence unit in Bernoulli Naive Bayes. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # binary word matrix.
y_b2 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # matching labels.
win_col_b2 = 1 # choose the word column for win.
print("win column:", X_b2[:, win_col_b2]) # inspect word presence.

▶ What you'll see: `win` appears only in spam rows in this tiny sample.

In [ ]:
p_win_ham_b2 = np.mean(X_b2[y_b2 == 0, win_col_b2]) # estimate p(win=1|ham).
p_win_spam_b2 = np.mean(X_b2[y_b2 == 1, win_col_b2]) # estimate p(win=1|spam).
print("p(win|ham):", p_win_ham_b2, "p(win|spam):", p_win_spam_b2) # inspect contrast.
assert p_win_ham_b2 == 0.0 and p_win_spam_b2 == 1.0 # verify unsmoothed extremes.
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], [p_win_ham_b2, p_win_spam_b2], color=["steelblue", "crimson"])
plt.ylim(0, 1.05); plt.title("Basic 2: p(win present | class)"); plt.show()

▶ What you'll see: the spam likelihood is 1 while the ham likelihood is 0 before smoothing.

👀 Takeaway: feature likelihoods are class-conditional frequencies.

### Basic 3 — Multiply a prior by one likelihood

**Goal.** Compute a one-feature class score, because Bayes' rule combines a prior with evidence. We build it in 2 steps.

In [ ]:
priors_b3 = np.array([0.625, 0.375]) # reuse ham/spam priors.
p_win_b3 = np.array([0.143, 0.800]) # use smoothed p(win=1|class) values.
print("priors:", priors_b3) # inspect base rates.
print("smoothed p(win|class):", p_win_b3) # inspect one likelihood per class.

▶ What you'll see: spam has the stronger word likelihood but the smaller prior.

In [ ]:
scores_b3 = priors_b3 * p_win_b3 # multiply p(y) by p(win|y).
post_b3 = normalize(scores_b3) # normalize scores.
print("scores:", np.round(scores_b3, 4)) # inspect unnormalized scores.
print("posterior:", np.round(post_b3, 3)) # inspect probabilities.
assert np.allclose(np.round(post_b3, 3), [0.230, 0.770]) # verify posterior.
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], post_b3, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.title("Basic 3: one-feature posterior"); plt.show()

▶ What you'll see: seeing `win` flips the posterior toward spam despite ham's larger prior.

👀 Takeaway: posterior odds are prior odds updated by likelihood evidence.

### Basic 4 — Include absent-word evidence

**Goal.** Use `1-p(x_j=1|y)` when a binary word is absent, because absence is still evidence in Bernoulli Naive Bayes. We build it in 2 steps.

In [ ]:
x_b4 = np.array([1, 1, 0]) # meeting and win are present; lunch is absent.
probs_b4 = np.array([[0.571, 0.143, 0.714], [0.400, 0.800, 0.400]]) # smoothed word-present probabilities.
print("x:", x_b4) # inspect word pattern.
print("p(word present|class):\n", probs_b4) # inspect likelihood table.

▶ What you'll see: the third feature is absent, so its contribution must use the complement.

In [ ]:
terms_b4 = np.where(x_b4 == 1, probs_b4, 1 - probs_b4) # choose present or absent terms.
print("selected terms:\n", np.round(terms_b4, 3)) # inspect per-feature evidence.
assert np.allclose(np.round(terms_b4[0], 3), [0.571, 0.143, 0.286]) # verify ham terms.
plt.figure(figsize=(5, 3)); plt.imshow(terms_b4, cmap="viridis", aspect="auto")
plt.colorbar(label="likelihood term"); plt.yticks([0, 1], ["ham", "spam"]); plt.xticks([0, 1, 2], ["meeting", "win", "not lunch"])
plt.title("Basic 4: present and absent evidence"); plt.show()

▶ What you'll see: absent lunch is more spam-like because spam has lower lunch-present probability.

👀 Takeaway: Bernoulli NB models both word presence and word absence.

### Basic 5 — Normalize two class scores

**Goal.** Convert proportional class scores into posterior probabilities, because probabilities must sum to one across classes. We build it in 2 steps.

In [ ]:
scores_b5 = np.array([0.01458, 0.07200]) # prior-times-product scores for ham and spam.
print("raw scores:", scores_b5) # inspect proportional scores.
print("raw score sum:", round(float(scores_b5.sum()), 5)) # inspect denominator.

▶ What you'll see: raw scores are not yet a probability distribution.

In [ ]:
post_b5 = normalize(scores_b5) # divide by total evidence.
print("posterior:", np.round(post_b5, 3)) # inspect normalized probabilities.
print("sum:", round(float(post_b5.sum()), 3)) # verify sum.
assert np.allclose(np.round(post_b5, 3), [0.168, 0.832]) # verify rounded posterior.
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], post_b5, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.title("Basic 5: normalized scores"); plt.show()

▶ What you'll see: normalization preserves the winner but makes the values interpretable as probabilities.

👀 Takeaway: Naive Bayes compares unnormalized scores, then normalizes them for probabilities.

### Basic 6 — Apply Laplace smoothing

**Goal.** Add pseudocounts to avoid zeros, because a never-seen word in one class should not make that class impossible. We build it in 3 steps.

In [ ]:
present_counts_b6 = np.array([[3, 0, 4], [1, 3, 1]]) # word-present counts in ham and spam.
class_sizes_b6 = np.array([5, 3]) # examples in each class.
alpha_b6 = 1.0 # add-one smoothing.
print("counts:\n", present_counts_b6) # inspect raw evidence.

▶ What you'll see: ham has zero observed `win` counts before smoothing.

In [ ]:
smoothed_b6 = (present_counts_b6 + alpha_b6) / (class_sizes_b6[:, None] + 2 * alpha_b6) # smooth binary probabilities.
print("smoothed p(word|class):\n", np.round(smoothed_b6, 3)) # inspect table.
assert np.allclose(np.round(smoothed_b6, 3), [[0.571, 0.143, 0.714], [0.4, 0.8, 0.4]]) # verify values.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(smoothed_b6, cmap="magma", aspect="auto")
plt.colorbar(label="p(word present | class)"); plt.yticks([0, 1], ["ham", "spam"]); plt.xticks([0, 1, 2], ["meeting", "win", "lunch"])
plt.title("Basic 6: smoothed likelihood table"); plt.show()

▶ What you'll see: all entries are strictly between 0 and 1 after smoothing.

👀 Takeaway: Laplace smoothing is a small regularizer for categorical probability tables.

### Basic 7 — Predict with a tiny helper

**Goal.** Use a reusable prediction helper, because Naive Bayes repeatedly performs the same prior × likelihood product. We build it in 2 steps.

In [ ]:
X_b7 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # toy matrix.
y_b7 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
classes_b7, priors_b7, probs_b7 = binary_nb_tables(X_b7, y_b7, alpha=1.0) # estimate NB tables.
print("priors:", np.round(priors_b7, 3)) # inspect priors.
print("probs:\n", np.round(probs_b7, 3)) # inspect likelihoods.

▶ What you'll see: the helper reproduces the smoothed tables from earlier examples.

In [ ]:
x_b7 = np.array([1, 1, 0]) # new message.
post_b7 = binary_nb_predict(x_b7, priors_b7, probs_b7) # compute posterior.
print("posterior:", np.round(post_b7, 3), "predicted class:", classes_b7[np.argmax(post_b7)]) # inspect prediction.
assert np.allclose(np.round(post_b7, 3), [0.168, 0.832]) # verify canonical posterior.
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], post_b7, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.title("Basic 7: helper prediction"); plt.show()

▶ What you'll see: the helper predicts spam with posterior about 0.832.

👀 Takeaway: the Bernoulli NB prediction rule is a few vectorized NumPy operations.

### Basic 8 — Compare probability product with log sum

**Goal.** Show that multiplying probabilities equals adding log probabilities, because log-space is the stable implementation. We build it in 2 steps.

In [ ]:
terms_b8 = np.array([0.4, 0.8, 0.6]) # three likelihood terms for one class.
product_b8 = float(np.prod(terms_b8)) # multiply in probability space.
log_sum_b8 = float(np.sum(np.log(terms_b8))) # add in log space.
print("product:", round(product_b8, 3)) # inspect product.
print("log sum:", round(log_sum_b8, 3)) # inspect log product.

▶ What you'll see: the log sum is the logarithm of the same product.

In [ ]:
recovered_b8 = float(np.exp(log_sum_b8)) # exponentiate log sum.
print("recovered product:", round(recovered_b8, 3)) # inspect equivalence.
assert round(recovered_b8, 3) == round(product_b8, 3) == 0.192 # verify equality.
plt.figure(figsize=(4, 3)); plt.bar(["product", "exp(log sum)"], [product_b8, recovered_b8], color=["gray", "seagreen"])
plt.title("Basic 8: product equals exp(log sum)"); plt.show()

▶ What you'll see: both bars have the same height.

👀 Takeaway: log-space changes the arithmetic, not the model's decision rule.

### Basic 9 — Build Gaussian likelihood pieces

**Goal.** Estimate class means and variances for continuous features, because Gaussian NB needs density parameters instead of word counts. We build it in 2 steps.

In [ ]:
X_b9 = np.array([[1.0, 2.0], [1.2, 1.8], [0.8, 2.2], [3.0, 0.8], [3.2, 1.0], [2.8, 0.7]]) # continuous features.
y_b9 = np.array([0, 0, 0, 1, 1, 1]) # labels.
means_b9 = np.vstack([X_b9[y_b9 == k].mean(axis=0) for k in [0, 1]]) # class means.
vars_b9 = np.vstack([X_b9[y_b9 == k].var(axis=0) + 1e-6 for k in [0, 1]]) # class variances.
print("means:\n", np.round(means_b9, 3)) # inspect centers.

▶ What you'll see: the two classes have clearly separated feature means.

In [ ]:
print("variances:\n", np.round(vars_b9, 3)) # inspect spreads.
assert np.allclose(np.round(means_b9, 3), [[1.0, 2.0], [3.0, 0.833]]) # verify means.
plt.figure(figsize=(4, 3)); plt.scatter(X_b9[y_b9 == 0, 0], X_b9[y_b9 == 0, 1], color="steelblue", label="class 0")
plt.scatter(X_b9[y_b9 == 1, 0], X_b9[y_b9 == 1, 1], color="crimson", label="class 1")
plt.title("Basic 9: continuous feature clusters"); plt.legend(); plt.show()

▶ What you'll see: each class forms a small cluster that a Gaussian density can summarize.

👀 Takeaway: Gaussian NB replaces categorical frequency tables with per-class means and variances.

### Basic 10 — Choose the lower decision score

**Goal.** Recompute the lesson's empirical score plus cost, because model selection should compare the full decision quantity. We build it in 3 steps.

In [ ]:
losses_b10 = np.array([0.213, 0.109, 0.454]) # verified per-example losses.
cost_b10 = 0.080 # method cost term.
raw_b10 = float(np.mean(losses_b10)) # empirical risk.
print("raw empirical risk:", round(raw_b10, 3)) # inspect R_S.
assert round(raw_b10, 3) == 0.259 # verify average.

▶ What you'll see: the raw average loss is 0.259.

In [ ]:
score_b10 = raw_b10 + cost_b10 # add cost.
alt_b10 = 0.379 # flexible alternative score.
stable_b10 = 0.80 * score_b10 # stabilized score.
print("scores:", np.round([score_b10, alt_b10, stable_b10], 3)) # inspect candidates.
assert round(score_b10, 3) == 0.339 and round(stable_b10, 3) == 0.271 # verify lesson values.

In [ ]:
labels_b10 = ["baseline+cost", "flexible", "stable"] # candidate names.
vals_b10 = np.array([score_b10, alt_b10, stable_b10]) # collect scores.
print("winner:", labels_b10[int(np.argmin(vals_b10))]) # identify lowest score.
plt.figure(figsize=(5, 3)); plt.bar(labels_b10, vals_b10, color=["steelblue", "orange", "seagreen"])
plt.ylabel("score lower is better"); plt.xticks(rotation=15); plt.title("Basic 10: full-score comparison"); plt.show()

▶ What you'll see: the stabilized score is the lowest of the three.

👀 Takeaway: selection should use the full score implied by the method, not the prettiest raw loss.

## 🟡 Easy

### Easy 1 — Train Bernoulli Naive Bayes end to end

**Goal.** Fit priors and smoothed likelihoods, then classify a new binary message, because this is the standard Bernoulli NB workflow. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # training features.
y_e1 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
words_e1 = np.array(["meeting", "win", "lunch"]) # feature names.
print("training shape:", X_e1.shape) # inspect dataset size.

▶ What you'll see: a tiny 8-example binary text dataset.

In [ ]:
classes_e1, priors_e1, probs_e1 = binary_nb_tables(X_e1, y_e1, alpha=1.0) # train smoothed tables.
print("classes:", classes_e1) # inspect class order.
print("priors:", np.round(priors_e1, 3)) # inspect priors.
print("likelihoods:\n", np.round(probs_e1, 3)) # inspect p(word present|class).

In [ ]:
x_e1 = np.array([1, 1, 0]) # new message.
post_e1 = binary_nb_predict(x_e1, priors_e1, probs_e1) # compute posterior.
pred_e1 = classes_e1[int(np.argmax(post_e1))] # choose max posterior class.
print("posterior:", np.round(post_e1, 3), "prediction:", pred_e1) # inspect prediction.
assert pred_e1 == 1 and np.allclose(np.round(post_e1, 3), [0.168, 0.832]) # verify expected result.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["ham", "spam"], post_e1, color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.title("Easy 1: Bernoulli NB posterior"); plt.ylabel("p(y|x)"); plt.show()

▶ What you'll see: the spam posterior is higher because `win` is strongly spam-associated.

👀 Takeaway: Bernoulli NB is count tables plus a Bayes normalization step.

### Easy 2 — Sweep smoothing strength

**Goal.** Vary alpha and watch the posterior change, because smoothing controls how strongly rare counts are trusted. We build it in 4 steps.

In [ ]:
X_e2 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # training features.
y_e2 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
x_e2 = np.array([1, 1, 0]) # message to classify.
alphas_e2 = np.array([0.1, 0.5, 1.0, 2.0, 10.0]) # smoothing strengths.
print("alphas:", alphas_e2) # inspect sweep values.

▶ What you'll see: the sweep runs from weak to strong smoothing.

In [ ]:
spam_posts_e2 = [] # store spam posterior for each alpha.
for alpha_e2 in alphas_e2:
    _, priors_e2, probs_e2 = binary_nb_tables(X_e2, y_e2, alpha=alpha_e2)
    post_e2 = binary_nb_predict(x_e2, priors_e2, probs_e2)
    spam_posts_e2.append(post_e2[1])
print("spam posteriors:", np.round(spam_posts_e2, 3)) # inspect smoothing effect.

In [ ]:
assert spam_posts_e2[0] > spam_posts_e2[-1] # stronger smoothing softens extreme evidence.
print("weak alpha posterior:", round(float(spam_posts_e2[0]), 3), "strong alpha posterior:", round(float(spam_posts_e2[-1]), 3))

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(alphas_e2, spam_posts_e2, marker="o", color="crimson")
plt.xscale("log"); plt.ylim(0, 1); plt.xlabel("alpha"); plt.ylabel("p(spam|x)")
plt.title("Easy 2: smoothing softens evidence"); plt.show()

▶ What you'll see: larger alpha pulls the spam posterior back toward the prior.

👀 Takeaway: smoothing is a stability knob that prevents tiny samples from sounding overconfident.

### Easy 3 — Compute log-space predictions for many messages

**Goal.** Classify several messages at once in log space, because real NB systems avoid long probability products. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # training matrix.
y_e3 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
_, priors_e3, probs_e3 = binary_nb_tables(X_e3, y_e3, alpha=1.0) # fit tables.
Xnew_e3 = np.array([[1, 1, 0], [0, 0, 1], [0, 1, 1], [1, 0, 1]]) # messages to classify.
print("new batch shape:", Xnew_e3.shape) # inspect batch size.

▶ What you'll see: four candidate messages, each with three binary features.

In [ ]:
posts_e3 = np.vstack([binary_nb_predict_log(row_e3, priors_e3, probs_e3) for row_e3 in Xnew_e3]) # classify every row.
preds_e3 = np.argmax(posts_e3, axis=1) # choose class labels.
print("posteriors:\n", np.round(posts_e3, 3)) # inspect probabilities.
print("predictions:", preds_e3) # inspect labels.

In [ ]:
assert np.array_equal(preds_e3, np.array([1, 0, 1, 0])) # verify predictions.
print("row sums:", np.round(posts_e3.sum(axis=1), 3)) # confirm posterior rows sum to one.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(posts_e3, cmap="viridis", aspect="auto")
plt.colorbar(label="posterior"); plt.xlabel("class"); plt.ylabel("message index")
plt.title("Easy 3: batch posterior heatmap"); plt.show()

▶ What you'll see: messages containing `win` lean spam, while ham-like messages lean class 0.

👀 Takeaway: log-space NB scales naturally from one row to a batch of rows.

### Easy 4 — Fit Gaussian Naive Bayes

**Goal.** Train Gaussian NB on continuous features and classify new points, because Naive Bayes is not limited to binary words. We build it in 4 steps.

In [ ]:
X_e4 = np.array([[1.0, 2.0], [1.2, 1.8], [0.8, 2.2], [3.0, 0.8], [3.2, 1.0], [2.8, 0.7]]) # continuous features.
y_e4 = np.array([0, 0, 0, 1, 1, 1]) # labels.
classes_e4 = np.array([0, 1]) # class order.
priors_e4 = np.array([np.mean(y_e4 == k) for k in classes_e4]) # priors.
print("priors:", priors_e4) # inspect priors.

▶ What you'll see: both classes have equal prior probability.

In [ ]:
means_e4 = np.vstack([X_e4[y_e4 == k].mean(axis=0) for k in classes_e4]) # Gaussian means.
vars_e4 = np.vstack([X_e4[y_e4 == k].var(axis=0) + 1e-6 for k in classes_e4]) # Gaussian variances.
print("means:\n", np.round(means_e4, 3)) # inspect centers.
print("vars:\n", np.round(vars_e4, 3)) # inspect spreads.

In [ ]:
Xtest_e4 = np.array([[1.1, 2.1], [2.9, 0.9], [2.0, 1.5]]) # test points.
log_posts_e4 = []
for x_e4 in Xtest_e4:
    log_density_e4 = -0.5 * (np.log(2 * np.pi * vars_e4) + ((x_e4 - means_e4) ** 2) / vars_e4)
    log_posts_e4.append(np.log(priors_e4) + log_density_e4.sum(axis=1))
log_posts_e4 = np.vstack(log_posts_e4)
shift_e4 = np.exp(log_posts_e4 - np.max(log_posts_e4, axis=1, keepdims=True))
posts_e4 = shift_e4 / shift_e4.sum(axis=1, keepdims=True)
print("posteriors:\n", np.round(posts_e4, 3))

In [ ]:
preds_e4 = np.argmax(posts_e4, axis=1) # choose predicted classes.
assert np.array_equal(preds_e4, np.array([0, 1, 0])) # verify classifications.
plt.figure(figsize=(4.5, 3.4)); plt.scatter(X_e4[y_e4 == 0, 0], X_e4[y_e4 == 0, 1], color="steelblue", label="class 0")
plt.scatter(X_e4[y_e4 == 1, 0], X_e4[y_e4 == 1, 1], color="crimson", label="class 1")
plt.scatter(Xtest_e4[:, 0], Xtest_e4[:, 1], marker="*", s=160, color="black", label="test")
plt.title("Easy 4: Gaussian NB test points"); plt.legend(); plt.show()

▶ What you'll see: test stars near each cluster receive that cluster's class.

👀 Takeaway: Gaussian NB uses the same Bayes structure with density estimates instead of count tables.

### Easy 5 — Hold out validation examples

**Goal.** Train on part of the toy data and score held-out examples, because validation tells whether the probabilistic rule survived unseen cases. We build it in 4 steps.

In [ ]:
X_e5 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # full dataset.
y_e5 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
train_idx_e5 = np.array([0, 1, 2, 5, 6, 7]) # training rows.
val_idx_e5 = np.array([3, 4]) # held-out rows.
print("train rows:", train_idx_e5, "validation rows:", val_idx_e5) # inspect split.

▶ What you'll see: two ham examples are held out from training.

In [ ]:
_, priors_e5, probs_e5 = binary_nb_tables(X_e5[train_idx_e5], y_e5[train_idx_e5], alpha=1.0) # train on split.
val_posts_e5 = np.vstack([binary_nb_predict_log(row_e5, priors_e5, probs_e5) for row_e5 in X_e5[val_idx_e5]]) # predict validation rows.
val_preds_e5 = np.argmax(val_posts_e5, axis=1) # posterior to labels.
print("validation posteriors:\n", np.round(val_posts_e5, 3)) # inspect probabilities.
print("validation predictions:", val_preds_e5) # inspect labels.

In [ ]:
val_acc_e5 = float(np.mean(val_preds_e5 == y_e5[val_idx_e5])) # validation accuracy.
print("validation accuracy:", round(val_acc_e5, 3)) # inspect held-out score.
assert val_acc_e5 == 1.0 # verify both held-out examples are correct.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["correct", "incorrect"], [np.sum(val_preds_e5 == y_e5[val_idx_e5]), np.sum(val_preds_e5 != y_e5[val_idx_e5])], color=["seagreen", "crimson"])
plt.title("Easy 5: held-out validation results"); plt.ylabel("count"); plt.show()

▶ What you'll see: both validation examples are classified correctly in this tiny split.

👀 Takeaway: validation evaluates future-facing behavior, not just training-table arithmetic.

## 🔴 Advanced

### Advanced 1 — Compare multinomial and Bernoulli event models

**Goal.** Classify count-valued text with a multinomial model and compare it to binary Bernoulli evidence, because the event model defines what each feature means. We build it in 5 steps.

In [ ]:
X_a1 = np.array([[3, 0, 1], [2, 0, 0], [0, 0, 2], [1, 0, 2], [0, 3, 0], [1, 2, 0], [0, 2, 1]]) # word counts.
y_a1 = np.array([0, 0, 0, 0, 1, 1, 1]) # labels.
x_a1 = np.array([1, 2, 0]) # new count vector with two win tokens.
print("count matrix shape:", X_a1.shape) # inspect dataset size.

▶ What you'll see: feature values can exceed 1, so frequency matters.

In [ ]:
alpha_a1 = 1.0
class_counts_a1 = np.array([np.sum(y_a1 == k) for k in [0, 1]])
priors_a1 = class_counts_a1 / class_counts_a1.sum()
word_counts_a1 = np.vstack([X_a1[y_a1 == k].sum(axis=0) for k in [0, 1]])
word_probs_a1 = (word_counts_a1 + alpha_a1) / (word_counts_a1.sum(axis=1, keepdims=True) + alpha_a1 * X_a1.shape[1])
print("multinomial word probs:\n", np.round(word_probs_a1, 3))

In [ ]:
log_scores_multi_a1 = np.log(priors_a1) + x_a1 @ np.log(word_probs_a1).T
post_multi_a1 = np.exp(log_scores_multi_a1 - np.max(log_scores_multi_a1)); post_multi_a1 = post_multi_a1 / post_multi_a1.sum()
print("multinomial posterior:", np.round(post_multi_a1, 3))

In [ ]:
Xbin_a1 = (X_a1 > 0).astype(float)
xbin_a1 = (x_a1 > 0).astype(float)
_, priors_bin_a1, probs_bin_a1 = binary_nb_tables(Xbin_a1, y_a1, alpha=1.0)
post_bin_a1 = binary_nb_predict_log(xbin_a1, priors_bin_a1, probs_bin_a1)
print("Bernoulli posterior:", np.round(post_bin_a1, 3))

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["multi ham", "multi spam"], post_multi_a1, color=["steelblue", "crimson"], alpha=0.8)
plt.bar(["bern ham", "bern spam"], post_bin_a1, color=["steelblue", "crimson"], alpha=0.45)
plt.ylim(0, 1); plt.title("Advanced 1: count vs presence evidence"); plt.xticks(rotation=15); plt.show()

▶ What you'll see: repeated `win` tokens can make the multinomial posterior more extreme than the Bernoulli posterior.

👀 Takeaway: multinomial NB uses token counts, while Bernoulli NB uses only present/absent events.

### Advanced 2 — Tune smoothing with validation loss

**Goal.** Select alpha by validation negative log likelihood, because the best smoothing strength should be chosen on unseen data. We build it in 5 steps.

In [ ]:
X_a2 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # dataset.
y_a2 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
train_a2 = np.array([0, 1, 2, 5, 6, 7]) # training indices.
val_a2 = np.array([3, 4]) # validation indices.
alphas_a2 = np.array([0.05, 0.1, 0.5, 1.0, 2.0, 5.0]) # candidates.
print("alpha candidates:", alphas_a2) # inspect sweep.

▶ What you'll see: several regularization strengths are evaluated.

In [ ]:
val_nll_a2 = []
for alpha_a2 in alphas_a2:
    _, priors_a2, probs_a2 = binary_nb_tables(X_a2[train_a2], y_a2[train_a2], alpha=alpha_a2)
    posts_a2 = np.vstack([binary_nb_predict_log(row_a2, priors_a2, probs_a2) for row_a2 in X_a2[val_a2]])
    true_probs_a2 = posts_a2[np.arange(len(val_a2)), y_a2[val_a2]]
    val_nll_a2.append(float(-np.mean(np.log(true_probs_a2))))
print("validation NLL:", np.round(val_nll_a2, 3))

In [ ]:
best_idx_a2 = int(np.argmin(val_nll_a2))
best_alpha_a2 = float(alphas_a2[best_idx_a2])
print("best alpha:", best_alpha_a2, "best NLL:", round(float(val_nll_a2[best_idx_a2]), 3))
assert best_alpha_a2 in alphas_a2

In [ ]:
raw_score_a2 = 0.259
cost_a2 = 0.080
full_score_a2 = raw_score_a2 + cost_a2
print("lesson full score:", round(full_score_a2, 3))
assert round(full_score_a2, 3) == 0.339

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(alphas_a2, val_nll_a2, marker="o", color="purple")
plt.xscale("log"); plt.axvline(best_alpha_a2, color="red", linestyle="--", label=f"best α={best_alpha_a2}")
plt.xlabel("alpha"); plt.ylabel("validation NLL"); plt.title("Advanced 2: tune smoothing on validation"); plt.legend(); plt.show()

▶ What you'll see: the selected alpha is the one with the lowest held-out log loss.

👀 Takeaway: smoothing is regularization, so it should be tuned with validation evidence.

### Advanced 3 — Inspect correlated-feature double counting

**Goal.** Show how duplicated features can overstate evidence, because Naive Bayes multiplies features as if they were independent. We build it in 5 steps.

In [ ]:
X_base_a3 = np.array([[1, 0], [1, 0], [0, 0], [0, 1], [0, 1], [1, 1]]) # two binary features.
y_a3 = np.array([0, 0, 0, 1, 1, 1]) # labels.
x_base_a3 = np.array([1, 0]) # test point.
print("base shape:", X_base_a3.shape) # inspect original feature count.

▶ What you'll see: the base dataset has two features.

In [ ]:
_, priors_base_a3, probs_base_a3 = binary_nb_tables(X_base_a3, y_a3, alpha=1.0)
post_base_a3 = binary_nb_predict_log(x_base_a3, priors_base_a3, probs_base_a3)
print("base posterior:", np.round(post_base_a3, 3))

In [ ]:
X_dup_a3 = np.column_stack([X_base_a3, X_base_a3[:, 0]]) # duplicate feature 0.
x_dup_a3 = np.array([1, 0, 1]) # duplicate the test feature too.
_, priors_dup_a3, probs_dup_a3 = binary_nb_tables(X_dup_a3, y_a3, alpha=1.0)
post_dup_a3 = binary_nb_predict_log(x_dup_a3, priors_dup_a3, probs_dup_a3)
print("duplicated-feature posterior:", np.round(post_dup_a3, 3))

In [ ]:
assert abs(post_dup_a3[0] - 0.5) > abs(post_base_a3[0] - 0.5) # duplicated evidence increases confidence.
print("confidence shift for class 0:", round(float(post_dup_a3[0] - post_base_a3[0]), 3))

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["base class0", "base class1"], post_base_a3, color=["steelblue", "crimson"], alpha=0.65)
plt.bar(["dup class0", "dup class1"], post_dup_a3, color=["steelblue", "crimson"], alpha=0.35)
plt.ylim(0, 1); plt.title("Advanced 3: duplicate feature double-counts evidence"); plt.xticks(rotation=15); plt.show()

▶ What you'll see: duplicating a correlated feature makes the posterior more confident even though no new information was added.

👀 Takeaway: the naive independence assumption can overcount redundant features.

### Advanced 4 — Calibrate a decision threshold with costs

**Goal.** Choose a spam threshold using false-positive and false-negative costs, because the best posterior cutoff depends on the decision problem. We build it in 5 steps.

In [ ]:
spam_prob_a4 = np.array([0.05, 0.12, 0.31, 0.44, 0.62, 0.71, 0.88, 0.96]) # predicted p(spam).
y_true_a4 = np.array([0, 0, 0, 1, 0, 1, 1, 1]) # true labels.
thresholds_a4 = np.array([0.3, 0.5, 0.7, 0.9]) # candidate thresholds.
print("thresholds:", thresholds_a4) # inspect thresholds.

▶ What you'll see: thresholds decide when posterior evidence is strong enough to mark spam.

In [ ]:
fp_cost_a4 = 5.0
fn_cost_a4 = 1.0
costs_a4 = []
for th_a4 in thresholds_a4:
    pred_a4 = (spam_prob_a4 >= th_a4).astype(int)
    fp_a4 = np.sum((pred_a4 == 1) & (y_true_a4 == 0))
    fn_a4 = np.sum((pred_a4 == 0) & (y_true_a4 == 1))
    costs_a4.append(fp_cost_a4 * fp_a4 + fn_cost_a4 * fn_a4)
print("costs:", costs_a4)

In [ ]:
best_idx_a4 = int(np.argmin(costs_a4))
best_th_a4 = float(thresholds_a4[best_idx_a4])
print("best threshold:", best_th_a4, "cost:", costs_a4[best_idx_a4])
assert best_th_a4 == 0.7

In [ ]:
pred_best_a4 = (spam_prob_a4 >= best_th_a4).astype(int)
print("best-threshold predictions:", pred_best_a4)
print("accuracy:", round(float(np.mean(pred_best_a4 == y_true_a4)), 3))

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(thresholds_a4, costs_a4, marker="o", color="darkorange")
plt.axvline(best_th_a4, color="red", linestyle="--", label="min cost")
plt.xlabel("spam threshold"); plt.ylabel("decision cost"); plt.title("Advanced 4: threshold chosen by cost"); plt.legend(); plt.show()

▶ What you'll see: the cost-minimizing threshold is higher than 0.5 because false positives are expensive.

👀 Takeaway: posteriors support decisions, but the threshold should reflect the cost of mistakes.

### Advanced 5 — Stress-test under class-prior shift

**Goal.** Recompute predictions when the deployment prior changes, because priors encode base rates that may differ between training and future data. We build it in 5 steps.

In [ ]:
X_a5 = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [1, 0, 1], [0, 0, 1], [0, 1, 0], [1, 1, 0], [0, 1, 1]]) # training matrix.
y_a5 = np.array([0, 0, 0, 0, 0, 1, 1, 1]) # labels.
_, train_priors_a5, probs_a5 = binary_nb_tables(X_a5, y_a5, alpha=1.0) # train likelihoods and original priors.
x_a5 = np.array([1, 1, 0]) # spam-like message.
print("training priors:", np.round(train_priors_a5, 3)) # inspect original base rates.

▶ What you'll see: training has ham as the majority class.

In [ ]:
shifted_priors_a5 = np.array([0.9, 0.1]) # deployment has much less spam.
post_train_prior_a5 = binary_nb_predict_log(x_a5, train_priors_a5, probs_a5) # predict with training priors.
post_shift_prior_a5 = binary_nb_predict_log(x_a5, shifted_priors_a5, probs_a5) # predict with shifted priors.
print("posterior with training prior:", np.round(post_train_prior_a5, 3))
print("posterior with shifted prior:", np.round(post_shift_prior_a5, 3))

In [ ]:
assert post_shift_prior_a5[1] < post_train_prior_a5[1] # lower spam base rate reduces spam posterior.
print("spam posterior drop:", round(float(post_train_prior_a5[1] - post_shift_prior_a5[1]), 3))

In [ ]:
losses_a5 = np.array([0.213, 0.109, 0.454]) # verified lesson losses.
score_a5 = float(losses_a5.mean() + 0.080) # full score with cost.
stable_a5 = 0.80 * score_a5 # stabilized score.
print("score:", round(score_a5, 3), "stable:", round(stable_a5, 3)) # inspect quantities.
assert round(score_a5, 3) == 0.339 and round(stable_a5, 3) == 0.271 # verify lesson values.

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["train prior spam", "shifted prior spam"], [post_train_prior_a5[1], post_shift_prior_a5[1]], color=["crimson", "gray"])
plt.ylim(0, 1); plt.ylabel("p(spam|x)"); plt.title("Advanced 5: prior shift changes posterior"); plt.show()

▶ What you'll see: the same likelihood evidence becomes less spam-confident when the deployment prior says spam is rare.

👀 Takeaway: priors are not decoration; class base-rate shift changes posterior probabilities and should be monitored.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Naive Bayes multiplies feature likelihoods as if features were conditionally independent within each class.

Naive Bayes turns class priors and feature likelihoods into a posterior ranking. The independence assumption is strong, but the resulting classifier is fast, transparent, and often competitive. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

np.random.seed(7)
random.seed(7)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def reg_ladder():
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
    y1 = np.array([1.0, 2.9, 5.2, 7.1, 8.9, 11.2])
    rungs.append(("D1 hand 3-point line plus checks", x1, y1))

    x2, y2 = make_regression(n_samples=220, n_features=3, n_informative=3, noise=8.0, random_state=2)
    rungs.append(("D2 clean make_regression", x2, y2))

    rng = np.random.default_rng(3)
    x3, y3 = make_regression(n_samples=260, n_features=5, n_informative=4, noise=18.0, random_state=3)
    outliers = rng.choice(np.arange(y3.size), size=18, replace=False)
    y3[outliers] = y3[outliers] + rng.normal(0.0, 240.0, size=outliers.size)
    rungs.append(("D3 noisy/outlier regression", x3, y3))

    diabetes = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", diabetes.data, diabetes.target))

    poly = PolynomialFeatures(degree=2, include_bias=False)
    x5 = poly.fit_transform(diabetes.data)
    y5 = diabetes.target.copy()
    rungs.append(("D5 Diabetes expanded interactions (real, 65-D)", x5, y5))

    return rungs


def reg_mse(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    mse = mean_squared_error(y_te, preds)
    r2 = r2_score(y_te, preds)
    return float(mse), float(r2)


def lesson_score(losses, cost, alternative):
    raw = sum(losses) / len(losses)
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    stabilized = 0.80 * score
    return raw, score, gap, relative_gap, stabilized


def print_ladder_preview(rungs):
    for name, X, y in rungs:
        unique = np.unique(y)
        label = "classes=" + str(unique.size) if unique.size <= 20 else "target_range=" + str((float(np.min(y)), float(np.max(y))))
        print(f"{name:42s} X={X.shape} {label}")
        print("sample X", np.round(X[:3], 3))
        print("sample y", np.round(y[:6], 3))


def plot_regression_results(rungs, fitted, mses):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        x_axis = np.arange(y.size)
        order = np.argsort(X[:, 0])
        preds = fitted[idx]
        ax.scatter(x_axis[:80], y[:80], s=12, alpha=0.65, label="actual")
        ax.scatter(x_axis[:80], preds[:80], s=12, alpha=0.65, label="fit")
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
        if idx == 0:
            ax.legend(fontsize=7)
    axes[1, 0].plot(np.arange(1, 6), mses, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out MSE")
    axes[1, 0].set_title("MSE vs complexity")
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


def plot_classification_results(rungs, build_and_predict, accs, title):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        x_vis = x_scaled[:, :2]
        x_tr, x_te, y_tr, y_te = train_test_split(x_vis, y, test_size=0.4, random_state=0, stratify=y)
        preds = build_and_predict(x_tr, y_tr, x_te)
        ax.scatter(x_te[:, 0], x_te[:, 1], c=preds, cmap="tab10", s=15, alpha=0.75)
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
    axes[1, 0].plot(np.arange(1, 6), accs, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_ylim(0.0, 1.05)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out accuracy")
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


## The concept, built once on D1

The lesson formula is

$$p(y=k\mid x)\propto p(y=k)\prod_{j=1}^d p(x_j\mid y=k)$$

We first reproduce the lesson arithmetic exactly: losses 0.213, 0.109, and 0.454 give $R_S=0.776/3=0.259$. Adding cost 0.080 gives score 0.339; the alternative 0.379 leaves gap 0.040. The stabilized score is $0.80\times 0.339=0.271$.

In [ ]:
losses = np.array([0.213, 0.109, 0.454])
cost = 0.080
alternative = 0.379
raw_unrounded, _, _, _, _ = lesson_score(losses, cost, alternative)
raw = 0.259
score = raw + cost
gap = alternative - score
relative_gap = gap / alternative
stabilized = 0.80 * score

assert round(float(losses.sum()), 3) == 0.776
assert round(raw_unrounded, 3) == 0.259
assert round(raw, 3) == 0.259
assert round(score, 3) == 0.339
assert round(gap, 3) == 0.040
assert round(relative_gap, 3) == 0.106

print("raw risk", round(raw, 3))
print("score", round(score, 3))
print("gap", round(gap, 3))
print("relative gap", round(relative_gap, 3))
print("stabilized", round(stabilized, 3))

### Build the method on D1

The concept cell below uses inspectable D1 numbers to show the method's own math before the notebook scales to real estimators on D1-D5.

In [ ]:
def naive_bayes_method(X_train, y_train, X_query):
    X_train = np.asarray(X_train, dtype=float)
    X_query = np.asarray(X_query, dtype=float)
    classes = np.unique(y_train)
    log_scores = []

    for cls in classes:
        subset = X_train[y_train == cls]
        prior = math.log(subset.shape[0] / X_train.shape[0])
        mean = subset.mean(axis=0)
        var = subset.var(axis=0) + 1e-6
        log_likelihood = -0.5 * np.sum(np.log(2.0 * np.pi * var))
        log_likelihood = log_likelihood - 0.5 * np.sum(((X_query - mean) ** 2) / var, axis=1)
        log_scores.append(prior + log_likelihood)

    scores = np.vstack(log_scores).T
    return classes[np.argmax(scores, axis=1)]

X_train = np.array([[0.0, 0.1], [0.2, -0.1], [2.8, 3.0], [3.1, 2.9], [3.2, 3.3]])
y_train = np.array([0, 0, 1, 1, 1])
query = np.array([[3.0, 3.1]])
pred = naive_bayes_method(X_train, y_train, query)
assert int(pred[0]) == 1
print("posterior winner", pred[0])

## The dataset ladder

In [ ]:
classification_rungs = clf_ladder()
print_ladder_preview(classification_rungs)

## Run the same method across D1-D5

In [ ]:
def build_and_predict(x_tr, y_tr, x_te):
    model = GaussianNB()
    model.fit(x_tr, y_tr)
    return model.predict(x_te)

accuracies = []
print("rung | accuracy | name")
for idx, (name, X, y) in enumerate(classification_rungs, start=1):
    acc = clf_accuracy(build_and_predict, X, y)
    accuracies.append(acc)
    print(f"D{idx} | {acc:.3f} | {name}")

## Results visualization

In [ ]:
plot_classification_results(classification_rungs, build_and_predict, accuracies, "Naive Bayes accuracy vs complexity")

## Pitfall on D5: optimizing the raw term and forgetting the cost

In [ ]:
name, X, y = classification_rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)

raw_candidates = {}
cost_aware = {}
for label, model, cost_add in [
    ("simple", GaussianNB(var_smoothing=1e-12), 0.080),
    ("tempting_complex", GaussianNB(var_smoothing=1e-12), 0.150),
]:
    fitted = clone(model)
    fitted.fit(x_tr, y_tr)
    acc = accuracy_score(y_te, fitted.predict(x_te))
    raw_error = 1.0 - acc
    raw_candidates[label] = raw_error
    cost_aware[label] = raw_error + cost_add

raw_winner = min(raw_candidates, key=raw_candidates.get)
cost_winner = min(cost_aware, key=cost_aware.get)
print("raw errors", {key: round(value, 3) for key, value in raw_candidates.items()})
print("cost-aware scores", {key: round(value, 3) for key, value in cost_aware.items()})
print("raw winner", raw_winner)
print("cost-aware winner", cost_winner)
print("lesson gap check", round(0.379 - 0.339, 3))
assert round(0.379 - 0.339, 3) == 0.040

## Evaluate it + practice

- Report the held-out accuracy beside a no-skill baseline such as majority-class accuracy or mean-target MSE.
- Sanity check that shuffling labels or targets destroys the useful signal.
- Ablation: remove scaling or increase Gaussian smoothing and compare D5 accuracy.
- Watch failure signals: validation instability, scale leakage, and a score that improves only when the lesson cost is ignored.
- Recompute the lesson raw risk, cost, gap, relative gap, and stabilized score whenever you compare settings.

Practice 1: change one hyperparameter and rerun the D1-D5 table.

Practice 2: add a label/target shuffle baseline and explain the drop.

Practice 3: repeat the pitfall cell with a different cost value.